In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

#  Load model
model_path = "/content/drive/MyDrive/toxic"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

#  Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

In [ ]:
# More harsh test examples
test_inputs = [
    "You are completely useless and nobody likes you.",
    "This is the dumbest thing I have ever seen.",
    "You always ruin everything.",
    "I hate you so much.",
    "You are a total failure in life.",
    "This is garbage work.",
    "suck my ass",
    # "bastard",
    "you bloody fucking idiot "
]

# Generate outputs
for text in test_inputs:
    prompt =  text

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=50)

    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("\n" + "="*50)
    print("Input:      ", text)
    print("Output:     ", pred)


Input:       You are completely useless and nobody likes you.
Output:      Non-toxic: Nobody likes you.

Input:       This is the dumbest thing I have ever seen.
Output:      Non-toxic: This is the worst thing I have ever seen.

Input:       You always ruin everything.
Output:      You always ruin everything.

Input:       I hate you so much.
Output:      I hate you so much.

Input:       You are a total failure in life.
Output:      Non-toxic: You are not good in life.

Input:       This is garbage work.
Output:      Non-toxic: This is not good work.

Input:       suck my ass
Output:      Non-toxic: I don't care.

Input:       you bloody fucking idiot 
Output:      Non-toxic: You are not good


In [ ]:
!pip install -q datasets

In [ ]:


from datasets import load_dataset

# Load ParaDetox
dataset = load_dataset("s-nlp/paradetox")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['en_toxic_comment', 'en_neutral_comment'],
        num_rows: 19744
    })
})


In [ ]:
def format_for_gpt(example):
    return {
        "prompt": example["prompt"],      # already correct
        "chosen": example["chosen"],
        "rejected": example["rejected"]
    }

dpo_gpt_dataset = dpo_final_dataset.map(format_for_gpt)

Map:   0%|          | 0/2921 [00:00<?, ? examples/s]

In [ ]:
print(dataset.column_names)

{'train': ['en_toxic_comment', 'en_neutral_comment']}


In [ ]:
def generate_rejected(dataset, model, tokenizer, device, max_samples=2000):
    prompts = []
    rejected_outputs = []

    for i in range(min(len(dataset), max_samples)):
        toxic = dataset[i]["en_toxic_comment"]

        prompt = f"Rewrite the following sentence in a positive way:\n{toxic}"

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            padding=True
        ).to(device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_length=64,
                do_sample=True,
                temperature=1.2,
                top_p=0.8,
                top_k=40,
                repetition_penalty=1.1
            )

        generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

        #  CLEAN BAD PREFIXES
        generated_text = generated_text.replace("Non-toxic:", "").strip()

        prompts.append(prompt)
        rejected_outputs.append(generated_text)

    return prompts, rejected_outputs

In [ ]:
prompts, rejected_outputs = generate_rejected(
    dataset["train"],
    model,
    tokenizer,
    device,
    max_samples=50
)

for i in range(5):
    print("\n--- Sample", i, "---")
    print("Prompt:", prompts[i])
    print("Rejected:", rejected_outputs[i])


--- Sample 0 ---
Prompt: Rewrite the following sentence in a positive way:
he had steel balls too !
Rejected: He had metal bars too.

--- Sample 1 ---
Prompt: Rewrite the following sentence in a positive way:
dude should have been taken to api , he would be right at home with all the other knuckleheads there
Rejected: dude should have been took to api, he would be right at home with all the other man there.

--- Sample 2 ---
Prompt: Rewrite the following sentence in a positive way:
im not gonna sell the fucking picture , i just want to contribute to the fucking article .
Rejected: Rewrite the following sentence in a positive way: I’m not gonna sold the picture, I just want to contribute to the article.

--- Sample 3 ---
Prompt: Rewrite the following sentence in a positive way:
the garbage that is being created by cnn and other news agencies is outrageous .
Rejected: The things that is being created by cnn and other news agency is terrible.

--- Sample 4 ---
Prompt: Rewrite the followi

In [ ]:
for i in range(3):
    print("\n--- Sample", i, "---")
    print("Prompt:", prompts[i])
    print("Rejected (model):", rejected_outputs[i])


--- Sample 0 ---
Prompt: Rewrite the following sentence in a positive way:
he had steel balls too !
Rejected (model): he had steel balls too

--- Sample 1 ---
Prompt: Rewrite the following sentence in a positive way:
dude should have been taken to api , he would be right at home with all the other knuckleheads there
Rejected (model): I will be right at home with all the other people there

--- Sample 2 ---
Prompt: Rewrite the following sentence in a positive way:
im not gonna sell the fucking picture , i just want to contribute to the fucking article .
Rejected (model): You’re not gonna sell the picture, I just want to contribute to the article.


In [ ]:
from datasets import Dataset

def build_dpo_dataset(dataset, max_samples=3000):
    data = []

    for i in range(min(len(dataset), max_samples)):
        toxic = dataset[i]["en_toxic_comment"].strip()
        chosen = dataset[i]["en_neutral_comment"].strip()

        if not toxic or not chosen:
            continue

        rejected = degrade_sentence(chosen)

        if chosen.lower() == rejected.lower():
            continue

        prompt = f"Rewrite the following sentence in a positive way:\n{toxic}"

        data.append({
            "prompt": prompt,
            "chosen": chosen,
            "rejected": rejected
        })

    return Dataset.from_list(data)

dpo_final_dataset = build_dpo_dataset(dataset["train"], max_samples=3000)

In [ ]:
import random

def degrade_sentence(sentence):
    words = sentence.split()

    if len(words) < 4:
        return sentence

    # randomly remove some words
    new_words = [w for w in words if random.random() > 0.2]

    # shuffle slightly
    if len(new_words) > 3:
        i = random.randint(0, len(new_words)-2)
        new_words[i], new_words[i+1] = new_words[i+1], new_words[i]

    return " ".join(new_words)

In [ ]:
print("Final dataset size:", len(dpo_final_dataset))

Final dataset size: 2921


In [ ]:
for i in range(3):
    print(f"\n--- Sample {i} ---")
    print("Prompt:", dpo_final_dataset[i]["prompt"])
    print("Chosen:", dpo_final_dataset[i]["chosen"])
    print("Rejected:", dpo_final_dataset[i]["rejected"])


--- Sample 0 ---
Prompt: Rewrite the following sentence in a positive way:
he had steel balls too !
Chosen: he was brave too!
Rejected: he brave too!

--- Sample 1 ---
Prompt: Rewrite the following sentence in a positive way:
dude should have been taken to api , he would be right at home with all the other knuckleheads there
Chosen: It would have been good if he went to api. He would fit in.
Rejected: would It have good he went He would fit in.

--- Sample 2 ---
Prompt: Rewrite the following sentence in a positive way:
im not gonna sell the fucking picture , i just want to contribute to the fucking article .
Chosen: I'm not gonna sell the picture, i just want to contribute to the article.
Rejected: I'm gonna sell the picture, just to want contribute the


In [ ]:
!pip install -q trl peft accelerate bitsandbytes

In [ ]:
from transformers import TrainingArguments
from trl import DPOTrainer
from peft import LoraConfig, get_peft_model

In [ ]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q", "v"],   # for T5 attention
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 884,736 || all params: 248,462,592 || trainable%: 0.3561


In [ ]:
output_dir = "/content/drive/MyDrive/dpo_modellatest"

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=100,
    fp16=True,
    remove_unused_columns=False,
    report_to="none"
)

In [ ]:
from trl import DPOTrainer, DPOConfig

In [ ]:
dpo_config = DPOConfig(
    output_dir="/content/drive/MyDrive/dpo_latest",

    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,

    learning_rate=5e-5,
    num_train_epochs=1,

    logging_steps=10,
    save_steps=100,

    fp16=True,
    remove_unused_columns=False,
    report_to="none",

    beta=0.1,
    max_length=128
)

In [ ]:
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=dpo_dataset,
    processing_class=tokenizer
)

Adding EOS to train dataset:   0%|          | 0/2921 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2921 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
dpo_trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Step,Training Loss
10,0.449213
20,0.128613
30,0.141427
40,0.027265
50,0.075108
60,0.025297
70,0.022307
80,0.017450
90,0.006595
100,0.042313


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=366, training_loss=0.04127830047862696, metrics={'train_runtime': 295.567, 'train_samples_per_second': 9.883, 'train_steps_per_second': 1.238, 'total_flos': 123058340352000.0, 'train_loss': 0.04127830047862696})

In [ ]:
def generate(text):
    inputs = tokenizer(text, return_tensors="pt").to(device)

    output = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,
        num_beams=5,
        early_stopping=True,
        pad_token_id=tokenizer.eos_token_id
    )

    result = tokenizer.decode(output[0], skip_special_tokens=True)
    result = result.replace(text, "").strip()

    return result

In [ ]:
test_inputs = [
    "You are stupid",
    "I hate you",
    "This is the worst thing ever",
    "You are useless",
    "Nobody likes you",
    "This code is trash",
    "You don’t understand anything",
    "This is completely wrong",
    "You are a failure",
    "Shut up, idiot"
]
for i, text in enumerate(test_inputs):
    prompt = f"Positive: {text}"
    output = generate(prompt)

    print(f"\n======== Example {i+1} ========")
    print(" Toxic Input  :", text)
    print("Model Output :", output)




======== Example 1 ========
Toxic Input  : You are stupid
 Model Output : You can improve your understanding

======== Example 2 ========
Toxic Input  : I hate you
 Model Output : I feel upset with you

======== Example 3 ========
Toxic Input  : This is the worst thing ever
 Model Output : This could be improved

======== Example 4 ========
Toxic Input  : You are useless
 Model Output : You have potential

======== Example 5 ========
Toxic Input  : Nobody likes you
 Model Output : You can build better connections

======== Example 6 ========
Toxic Input  : This code is trash
 Model Output : This code can be enhanced

======== Example 7 ========
Toxic Input  : You don’t understand anything
 Model Output : You can learn this step by step

======== Example 8 ========
Toxic Input  : This is completely wrong
 Model Output : This needs some correction

======== Example 9 ========
Toxic Input  : You are a failure
 Model Output : You can succeed with effort

======== Example 10 ========
Toxic